In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader
from datasets import Dataset as HFDataset  # 引入 Hugging Face 的 Dataset
from transformers import BertTokenizer
from tqdm import tqdm

In [2]:

# 加载和预处理数据
print("加载数据中...")
data = pd.read_csv("movie_reviews/movie_reviews.csv")  # 替换为你的数据集路径
# 创建 Hugging Face 数据集
hf_dataset = HFDataset.from_pandas(data)

加载数据中...


In [3]:

# 使用预训练的BERT分词器
print("加载分词器中...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

加载分词器中...


In [9]:

# 定义最大序列长度
max_len = 100

# 定义分词函数
def tokenize_function(example):
    return tokenizer(
        example["text"],
        max_length=max_len,
        truncation=True,
        padding="max_length"
    )
    

In [ ]:
# 对文本进行分词，并显示进度条
print("编码文本中...")
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True, desc="Tokenizing dataset")

In [29]:
tokenized_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 40000
})

In [28]:

# 转换为 PyTorch 张量
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

编码文本中...


Tokenizing dataset:   0%|          | 0/40000 [00:00<?, ? examples/s]

In [14]:

# 划分训练集和测试集
print("划分数据集中...")
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

划分数据集中...


In [15]:

# 使用 DataLoader 加载数据
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [16]:

# 定义GRU模型
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)  # 嵌入层
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)  # GRU层，batch_first=True表示输入输出的第一个维度是batch大小
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),  # 全连接层
            nn.ReLU(),  # 激活函数
            nn.Dropout(0.5),  # 随机失活防止过拟合
            nn.Linear(64, output_dim),  # 最终输出层
            nn.Sigmoid()  # 将输出值限制在0到1之间
        )

    def forward(self, x):
        x = self.embedding(x)  # 嵌入层处理输入
        _, hidden = self.gru(x)  # GRU处理后获取隐藏状态
        out = self.fc(hidden.squeeze(0))  # 将隐藏状态传入全连接层
        return out
        

In [17]:
# 初始化模型
print("初始化模型中...")
vocab_size = tokenizer.vocab_size  # 词汇表大小
embedding_dim = 128  # 嵌入向量维度
hidden_dim = 128  # GRU隐藏层维度
output_dim = 1  # 输出维度（单个分类任务）

model = GRUClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)  # 初始化模型

# 损失函数和优化器
criterion = nn.BCELoss()  # 二元交叉熵损失函数
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam优化器

初始化模型中...


In [18]:


# 评估模型
def evaluate_model(model, test_loader):
    print("评估模型中...")
    model.eval()  # 设置模型为评估模式
    all_preds = []
    all_labels = []
    with torch.no_grad():  # 禁用梯度计算以提高评估效率
        for batch in tqdm(test_loader, desc="评估中"):
            input_ids = batch['input_ids']  # 获取输入ID
            labels = batch['label'].float()  # 获取标签
            outputs = model(input_ids).squeeze()  # 模型预测结果
            preds = (outputs > 0.5).int()  # 将概率转换为二分类标签
            all_preds.extend(preds.tolist())  # 收集预测结果
            all_labels.extend(labels.tolist())  # 收集真实标签
    accuracy = accuracy_score(all_labels, all_preds)  # 计算准确率
    print("准确率:", accuracy)
    print(classification_report(all_labels, all_preds))  # 打印分类报告
    return accuracy
    

#from torch.utils.tensorboard import SummaryWriter

# 训练模型
epochs = 5  # 训练轮数
def train_and_evaluate(model, train_loader, test_loader):
    print("开始训练模型...")
    # 创建一个 TensorBoard 记录器
    #writer = SummaryWriter()

    global_step = 0  # 用于记录当前的训练步骤（batch）
    
    for epoch in range(epochs):
        model.train()  # 设置模型为训练模式
        total_loss = 0  # 累计当前 epoch 的损失

        for batch in tqdm(train_loader, desc=f"训练第 {epoch+1} 轮"):
            optimizer.zero_grad()  # 清零梯度
            input_ids = batch['input_ids']  # 获取输入 ID
            labels = batch['label'].float()  # 获取标签并转换为 float32

            outputs = model(input_ids)  # 前向传播
            loss = criterion(outputs.squeeze(), labels)  # 计算损失
            loss.backward()  # 反向传播
            optimizer.step()  # 更新参数

            total_loss += loss.item()  # 累加损失

            # 每个 batch 的损失写入 TensorBoard
            #writer.add_scalar('Loss/train', loss.item(), global_step)
            global_step += 1  # 增加全局训练步数

        # 打印每个 epoch 的总损失
        print(f"第 {epoch+1}/{epochs} 轮, 损失: {total_loss:.4f}")

        # 每轮训练后进行评估
        accuracy = evaluate_model(model, test_loader)

        # 每4轮保存一次模型
        if (epoch + 1) % 4 == 0:
            torch.save(model.state_dict(), f"gru_text_classifier_epoch_{epoch+1}.pth")  # 保存模型权重
            print(f"模型已保存: gru_text_classifier_epoch_{epoch+1}.pth")

    # 训练完成后关闭 TensorBoard 记录器
    #writer.close()

train_and_evaluate(model, train_loader, test_loader)

开始训练模型...


训练第 1 轮: 100%|██████████| 1000/1000 [01:08<00:00, 14.53it/s]


第 1/5 轮, 损失: 692.4892
评估模型中...


评估中: 100%|██████████| 250/250 [00:03<00:00, 74.48it/s]


准确率: 0.500875
              precision    recall  f1-score   support

         0.0       0.64      0.02      0.03      4019
         1.0       0.50      0.99      0.66      3981

    accuracy                           0.50      8000
   macro avg       0.57      0.50      0.35      8000
weighted avg       0.57      0.50      0.35      8000



...

准确率: 0.831125
              precision    recall  f1-score   support

         0.0       0.84      0.82      0.83      4019
         1.0       0.82      0.84      0.83      3981

    accuracy                           0.83      8000
   macro avg       0.83      0.83      0.83      8000
weighted avg       0.83      0.83      0.83      8000

模型已保存: gru_text_classifier_epoch_4.pth


训练第 5 轮: 100%|██████████| 1000/1000 [01:12<00:00, 13.74it/s]


第 5/5 轮, 损失: 187.9464
评估模型中...


评估中: 100%|██████████| 250/250 [00:03<00:00, 71.27it/s]

准确率: 0.8245
              precision    recall  f1-score   support

         0.0       0.82      0.84      0.83      4019
         1.0       0.83      0.81      0.82      3981

    accuracy                           0.82      8000
   macro avg       0.82      0.82      0.82      8000
weighted avg       0.82      0.82      0.82      8000



In [36]:

# 加载测试数据
test_data = pd.read_csv("test_data.csv")
# 创建 Hugging Face 数据集
hf_test_dataset = HFDataset.from_pandas(test_data)

In [37]:

# 对文本进行分词，并显示进度条
print("编码文本中...")
tokenized_test_dataset = hf_test_dataset.map(tokenize_function, batched=True, desc="Tokenizing test_dataset")

编码文本中...


Tokenizing test_dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [48]:
# 转换为 PyTorch 张量
tokenized_test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", 'Id'])

test_loader = DataLoader(tokenized_test_dataset, batch_size=32)

In [49]:

# 初始化模型
print("加载模型中...")
vocab_size = tokenizer.vocab_size  # 分词器的词汇大小
embedding_dim = 128  # 嵌入层维度
hidden_dim = 128  # GRU隐藏层维度
output_dim = 1  # 输出维度（分类任务）

model = GRUClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)  # 初始化模型
model.load_state_dict(torch.load("gru_text_classifier_epoch_4.pth"))  # 加载模型权重
model.eval()  # 设置模型为评估模式

加载模型中...


GRUClassifier(
  (embedding): Embedding(30522, 128)
  (gru): GRU(128, 128, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

In [50]:

# 推理并生成结果
print("开始推理...")
results = []
with torch.no_grad():  # 禁用梯度计算，提升推理效率
    for batch in tqdm(test_loader, desc="推理中"):

        input_ids = batch['input_ids']  # 获取输入ID
        ids = batch['Id']  # 获取样本ID
        outputs = model(input_ids).squeeze()  # 模型预测输出
        preds = (outputs > 0.5).int()  # 将概率值转换为二分类标签
        # results.extend(preds.tolist())  # 收集预测结果
        results.extend(zip(ids.tolist(), preds.tolist()))  # 收集预测结果
        
        
# 保存结果
print("保存结果中...")
output_df = pd.DataFrame(results, columns=["Id", "Category"])  # 创建结果数据框
output_df.to_csv("submission.csv", index=False)  # 保存结果为CSV文件
print("结果已保存到 submission.csv")

开始推理...


推理中: 100%|██████████| 313/313 [00:03<00:00, 82.72it/s]

保存结果中...
结果已保存到 submission.csv
